In [5]:
import pandas as pd

fp = r"C:\dev\projects\heat_mortality_analysis\data\raw\calima_Heliyon\Envío_datos_Calima.xlsx"

xl = pd.ExcelFile(fp)
xl.sheet_names

['Hoja1']

In [10]:
df0 = pd.read_excel(fp, sheet_name=xl.sheet_names[0])
print(df0.shape, df0.columns.tolist())
df0.head(5)

(480, 9) ['Fecha ini', 'Fecha fin', 'año', 'mes', 'nº aero afectados', 'visibilidad media ', 'visibil min proc', 'D', 'DAI']


,Fecha ini,Fecha fin,año,mes,nº aero afectados,visibilidad media,visibil min proc,D,DAI
0,1980-03-20,1980-03-20,1980,3,2,10000,10000.0,1,0.333333
1,1980-04-04,1980-04-04,1980,4,2,8500,8000.0,1,0.392157
2,1980-05-20,1980-05-22,1980,5,5,7600,5000.0,2,2.192982
3,1980-06-07,1980-06-07,1980,6,2,9500,9000.0,1,0.350877
4,1980-07-04,1980-07-04,1980,7,2,9500,9000.0,1,0.350877


In [11]:
df0.dtypes
# columnas que parezcan fechas
[c for c in df0.columns if "date" in c.lower() or "fecha" in c.lower()]

['Fecha ini', 'Fecha fin']

In [12]:
# df_ep = dataframe de episodios (el del Excel)
df_ep = df0.copy()

df_ep["Fecha ini"] = pd.to_datetime(df_ep["Fecha ini"], errors="coerce")
df_ep["Fecha fin"] = pd.to_datetime(df_ep["Fecha fin"], errors="coerce")

# episodios que intersectan el rango 2016-01-01 a 2024-12-31
start = pd.Timestamp("2016-01-01")
end   = pd.Timestamp("2024-12-31")

df1 = df_ep[(df_ep["Fecha fin"] >= start) & (df_ep["Fecha ini"] <= end)].copy()

# opcional: recortar fechas al rango
df1["Fecha ini"] = df1["Fecha ini"].clip(lower=start)
df1["Fecha fin"] = df1["Fecha fin"].clip(upper=end)

df1 = df1.sort_values(["Fecha ini", "Fecha fin"]).reset_index(drop=True)

print(df1.shape)
display(df1.head())
display(df1.tail())

(45, 9)


,Fecha ini,Fecha fin,año,mes,nº aero afectados,visibilidad media,visibil min proc,D,DAI
0,2016-03-02,2016-03-03,2016,3,2,7500,6000.0,1,0.444444
1,2016-07-20,2016-07-21,2016,7,2,9000,9000.0,1,0.370370
2,2016-08-28,2016-08-28,2016,8,2,10000,10000.0,1,0.333333
3,2016-09-29,2016-09-29,2016,9,3,9000,9000.0,1,0.555556
4,2016-12-25,2016-12-26,2016,12,3,6666,4000.0,2,1.500150


,Fecha ini,Fecha fin,año,mes,nº aero afectados,visibilidad media,visibil min proc,D,DAI
40,2022-01-14,2022-01-17,2022,1,6,6158,600.0,2,3.247808
41,2022-01-19,2022-01-21,2022,1,5,7530,5000.0,2,2.213369
42,2022-01-28,2022-01-29,2022,1,6,1200,1200.0,2,16.666667
43,2022-02-08,2022-02-09,2022,2,2,3000,3000.0,1,1.111111
44,2022-03-17,2022-03-18,2022,3,3,7000,7000.0,1,0.714286


In [13]:
import pandas as pd
import numpy as np

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

h = df1.copy()

# Duración del episodio (días)
h["Fecha ini"] = pd.to_datetime(h["Fecha ini"])
h["Fecha fin"] = pd.to_datetime(h["Fecha fin"])
h["dur_days"] = (h["Fecha fin"] - h["Fecha ini"]).dt.days + 1

# renombres cómodos
h = h.rename(columns={
    "nº aero afectados": "airports_n",
    "visibilidad media ": "vis_mean",
    "visibilidad media": "vis_mean",
    "visibil min proc": "vis_min_proc",
    "D": "D",
    "DAI": "DAI"
})

section("GLANCE (2016–2024 episodes)")
print("rows:", len(h), "| cols:", h.shape[1])
display(h[["Fecha ini","Fecha fin","dur_days","airports_n","vis_mean","vis_min_proc","D","DAI"]].head(10))

section("DURATION")
display(h["dur_days"].describe())
display(h["dur_days"].value_counts().sort_index())

section("CORE VARIABLE DISTRIBUTIONS")
display(h[["airports_n","vis_mean","vis_min_proc","D","DAI"]].describe())

section("CORRELATIONS (numeric)")
display(h[["dur_days","airports_n","vis_mean","vis_min_proc","D","DAI"]].corr(numeric_only=True))

section("SEVERITY GROUPS (D)")
display(
    h.groupby("D")[["dur_days","airports_n","vis_mean","vis_min_proc","DAI"]]
    .agg(["count","mean","median","min","max"])
)

section("TOP EXTREMES (by DAI)")
display(h.sort_values("DAI", ascending=False)[["Fecha ini","Fecha fin","dur_days","airports_n","vis_min_proc","D","DAI"]].head(10))


GLANCE (2016–2024 episodes)
rows: 45 | cols: 10


,Fecha ini,Fecha fin,dur_days,airports_n,vis_mean,vis_min_proc,D,DAI
0,2016-03-02,2016-03-03,2,2,7500,6000.0,1,0.444444
1,2016-07-20,2016-07-21,2,2,9000,9000.0,1,0.370370
2,2016-08-28,2016-08-28,1,2,10000,10000.0,1,0.333333
3,2016-09-29,2016-09-29,1,3,9000,9000.0,1,0.555556
4,2016-12-25,2016-12-26,2,3,6666,4000.0,2,1.500150
5,2017-01-02,2017-01-02,1,2,10000,10000.0,1,0.333333
6,2017-01-12,2017-01-13,2,5,8866,7000.0,1,0.939920
7,2017-03-10,2017-03-10,1,2,7500,6000.0,1,0.444444
8,2017-04-03,2017-04-03,1,3,7333,3000.0,1,0.681849
9,2017-06-25,2017-06-26,2,2,10000,10000.0,1,0.333333



DURATION


count    45.000000
mean      1.800000
std       0.814639
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       4.000000
Name: dur_days, dtype: float64

dur_days
1    18
2    20
3     5
4     2
Name: count, dtype: int64

...

,dur_days,airports_n,vis_mean,vis_min_proc,D,DAI
dur_days,1.000000,0.564311,-0.260752,-0.379495,0.760729,0.347070
airports_n,0.564311,1.000000,-0.459918,-0.543112,0.520981,0.599332
vis_mean,-0.260752,-0.459918,1.000000,0.882145,-0.362852,-0.693103
vis_min_proc,-0.379495,-0.543112,0.882145,1.000000,-0.426514,-0.569872
D,0.760729,0.520981,-0.362852,-0.426514,1.000000,0.550213
DAI,0.347070,0.599332,-0.693103,-0.569872,0.550213,1.000000



SEVERITY GROUPS (D)


dur_days                          airports_n                           ...  \
     count      mean median min max      count      mean median min max  ...   
D                                                                        ...   
1       35  1.485714    1.0   1   2         35  2.828571    2.0   2   6  ...   
2        9  2.777778    3.0   2   4          9  4.333333    5.0   2   6  ...   
3        1  4.000000    4.0   4   4          1  6.000000    6.0   6   6  ...   

  vis_min_proc                                         DAI            \
         count         mean  median     min      max count      mean   
D                                                                      
1           35  7000.000000  7000.0  1100.0  10000.0    35  0.641297   
2            9  4388.888889  4000.0   600.0   9000.0     9  4.321551   
3            1  3000.000000  3000.0  3000.0   3000.0     1  4.390458   

                                  
     median       min        max  
D                                 
1  0.512821  0.333333   2.318841  
2  2.213369  0.701754  16.666667  
3  4.390458  4.390458   4.390458  

[3 rows x 25 columns]


TOP EXTREMES (by DAI)


,Fecha ini,Fecha fin,dur_days,airports_n,vis_min_proc,D,DAI
42,2022-01-28,2022-01-29,2,6,1200.0,2,16.666667
30,2020-02-22,2020-02-24,3,6,700.0,2,7.794232
11,2017-08-06,2017-08-09,4,6,3000.0,3,4.390458
34,2021-02-16,2021-02-18,3,4,3000.0,2,3.864734
40,2022-01-14,2022-01-17,4,6,600.0,2,3.247808
19,2018-03-28,2018-03-28,1,4,1100.0,1,2.318841
41,2022-01-19,2022-01-21,3,5,5000.0,2,2.213369
23,2019-02-04,2019-02-05,2,5,8000.0,2,2.164502
22,2018-12-24,2018-12-25,2,6,5000.0,1,1.538462
4,2016-12-25,2016-12-26,2,3,4000.0,2,1.500150


In [14]:
df1["Fecha ini"] = pd.to_datetime(df1["Fecha ini"])
df1["Fecha fin"] = pd.to_datetime(df1["Fecha fin"])
df1["dur_days"] = (df1["Fecha fin"] - df1["Fecha ini"]).dt.days + 1

## Heliyon calima episodes (Excel) — column meanings

- **Fecha ini**: Start date of the calima episode (inclusive).
- **Fecha fin**: End date of the calima episode (inclusive). **Note:** Data collection untill 18/03/2022
- **año**: Calendar year of the episode (typically derived from the start date).
- **mes**: Calendar month of the episode (typically derived from the start date).
- **nº aero afectados**: Number of airports affected during the episode (operational impact proxy).
- **visibilidad media**: Mean visibility during the episode (meters). Higher = clearer air.
- **visibil min proc**: Minimum visibility observed during the episode (meters). Lower = worse visibility / thicker dust.
- **D**: Episode severity category (ordinal). Higher D = more severe episode.
- **DAI**: Dust/Calima intensity index (continuous). Higher DAI = more intense episode (often combines duration + visibility degradation + spatial impact).
- **dur_days**: Episode duration in days = (Fecha fin − Fecha ini) + 1.

## Dust Adversity Index (DAI) — summary (Heliyon)

**Goal:** rank dust (calima) events by how “adverse” they are in a given area. The authors argue this is harder than ranking rain/temperature extremes because dust intensity depends on multiple dimensions.

**What DAI captures (three pillars):**
- **Visibility reduction** (proxy for higher dust concentration)
- **Spatial extent** (how many stations/airports are affected → less local, more widespread)
- **Persistence** (event duration in days)

**Why a new index:** existing severity indices often use only visibility, or AOD, or combinations like “affected stations + visibility”, but typically miss one or more of: concentration proxy, area affected, and duration.

**Definition (conceptual):**  
DAI multiplies:
1) **Fraction of affected stations** *(s / sT)*  
2) **Spatial comparability factor** *(x / xr)* (set to 1 in their study)  
3) **Duration factor** *(d)* (binned 1–5 using duration percentiles to reduce outlier influence)  
4) **Inverse of mean visibility** *(10^4 / Vm)* (lower Vm → higher DAI)

**Key terms:**
- **s**: number of affected stations (2–6 in their setup)
- **sT**: total stations in the area (6 in their setup)
- **d**: duration category (1–5) based on duration percentiles
- **Vm**: mean visibility across stations and over the full event (meters)
- **x/xr**: max station separation normalized by a reference distance (used to compare regions; = 1 in their case)

**Extra note:** they also use **K-means** later to cluster geopotential height anomaly patterns and categorize dust-event days.

# Desition: take DAI as variable core.
-Bin DAE and modelate

In [19]:
# Binning por percentiles (terciles): 

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

# df1 = episodios 2016–2024 (44 filas) que ya tienes
h = df1.copy()

# tipos
h["Fecha ini"] = pd.to_datetime(h["Fecha ini"], errors="coerce")
h["Fecha fin"] = pd.to_datetime(h["Fecha fin"], errors="coerce")
h["DAI"] = pd.to_numeric(h["DAI"], errors="coerce")

# opcional: duración del episodio
h["dur_days"] = (h["Fecha fin"] - h["Fecha ini"]).dt.days + 1

section("BINS BY PERCENTILES (TERCILES) — EPISODE FLAGS")
h["calima_flag_ep"] = pd.qcut(h["DAI"], q=3, labels=["green","yellow","red"])

display(h["calima_flag_ep"].value_counts().sort_index().to_frame("episodes"))

section("LOWEST DAI (sample)")
display(h.sort_values("DAI")[["Fecha ini","Fecha fin","DAI","dur_days","calima_flag_ep"]].head(10))

section("HIGHEST DAI (sample)")
display(h.sort_values("DAI", ascending=False)[["Fecha ini","Fecha fin","DAI","dur_days","calima_flag_ep"]].head(10))


BINS BY PERCENTILES (TERCILES) — EPISODE FLAGS


,episodes
calima_flag_ep,
green,16
yellow,14
red,15



LOWEST DAI (sample)


,Fecha ini,Fecha fin,DAI,dur_days,calima_flag_ep
2,2016-08-28,2016-08-28,0.333333,1,green
5,2017-01-02,2017-01-02,0.333333,1,green
9,2017-06-25,2017-06-26,0.333333,2,green
31,2020-03-08,2020-03-08,0.333333,1,green
26,2019-06-02,2019-06-02,0.350877,1,green
29,2020-02-07,2020-02-08,0.350877,2,green
12,2017-08-24,2017-08-25,0.350877,2,green
1,2016-07-20,2016-07-21,0.370370,2,green
20,2018-08-12,2018-08-12,0.370370,1,green
38,2021-10-16,2021-10-16,0.370370,1,green



HIGHEST DAI (sample)


,Fecha ini,Fecha fin,DAI,dur_days,calima_flag_ep
42,2022-01-28,2022-01-29,16.666667,2,red
30,2020-02-22,2020-02-24,7.794232,3,red
11,2017-08-06,2017-08-09,4.390458,4,red
34,2021-02-16,2021-02-18,3.864734,3,red
40,2022-01-14,2022-01-17,3.247808,4,red
19,2018-03-28,2018-03-28,2.318841,1,red
41,2022-01-19,2022-01-21,2.213369,3,red
23,2019-02-04,2019-02-05,2.164502,2,red
22,2018-12-24,2018-12-25,1.538462,2,red
4,2016-12-25,2016-12-26,1.500150,2,red


In [23]:
def section(t):
    print("\n" + "="*90); print(t); print("="*90)

# ep = h (episodios) con columnas: 'Fecha ini', 'Fecha fin', 'calima_flag_ep'
ep = h.copy()
ep["Fecha ini"] = pd.to_datetime(ep["Fecha ini"], errors="coerce")
ep["Fecha fin"] = pd.to_datetime(ep["Fecha fin"], errors="coerce")

rows = []
for _, r in ep.iterrows():
    d0, d1 = r["Fecha ini"], r["Fecha fin"]
    flag = r["calima_flag_ep"]
    if pd.isna(d0) or pd.isna(d1) or pd.isna(flag):
        continue
    for d in pd.date_range(d0, d1, freq="D"):
        rows.append({"date": d, "flag": str(flag)})

df_day = pd.DataFrame(rows)

print("daily rows:", len(df_day), "| range:", df_day["date"].min(), "->", df_day["date"].max())
df_day.head()

daily rows: 81 | range: 2016-03-02 00:00:00 -> 2022-03-18 00:00:00


,date,flag
0,2016-03-02,green
1,2016-03-03,green
2,2016-07-20,green
3,2016-07-21,green
4,2016-08-28,green


In [24]:
def section(t):
    print("\n" + "="*90); print(t); print("="*90)

# 0) master weeks (para alinear)
parquet_path = r"C:\dev\projects\heat_mortality_analysis\data\processed\eda_ready\tfe\eda_ready_weekly_tfe.parquet"
master_weeks = pd.read_parquet(parquet_path, columns=["week_start"]).copy()
master_weeks["week_start"] = pd.to_datetime(master_weeks["week_start"])
master_weeks = master_weeks.sort_values("week_start").drop_duplicates().reset_index(drop=True)

# 1) episodios (h) -> días
ep = h.copy()
ep["Fecha ini"] = pd.to_datetime(ep["Fecha ini"], errors="coerce")
ep["Fecha fin"] = pd.to_datetime(ep["Fecha fin"], errors="coerce")

rows = []
for _, r in ep.iterrows():
    d0, d1 = r["Fecha ini"], r["Fecha fin"]
    flag = r["calima_flag_ep"]
    if pd.isna(d0) or pd.isna(d1) or pd.isna(flag):
        continue
    for d in pd.date_range(d0, d1, freq="D"):
        rows.append({"date": d, "flag": str(flag)})

df_day = pd.DataFrame(rows)
df_day["week_start"] = (df_day["date"] - pd.to_timedelta(df_day["date"].dt.weekday, unit="D")).dt.normalize()

# 2) week flag = max severity in that week
flag_to_score = {"green": 1, "yellow": 2, "red": 3}
score_to_flag = {1: "green", 2: "yellow", 3: "red"}

df_day["score"] = df_day["flag"].map(flag_to_score)

weekly_flags = (
    df_day.groupby("week_start")
    .agg(score=("score", "max"))
    .reset_index()
)
weekly_flags["calima_dai_flag"] = weekly_flags["score"].map(score_to_flag)
weekly_flags = weekly_flags[["week_start","calima_dai_flag"]]

# 3) merge to master weeks
calima_weekly = master_weeks.merge(weekly_flags, on="week_start", how="left")

# 4) Fill blue ONLY where Heliyon coverage exists; after end-of-coverage -> NaN
heliyon_end = pd.Timestamp("2022-03-31")
heliyon_end_week = (heliyon_end - pd.to_timedelta(heliyon_end.weekday(), unit="D")).normalize()

calima_weekly["calima_dai_flag"] = np.where(
    calima_weekly["week_start"] <= heliyon_end_week,
    calima_weekly["calima_dai_flag"].fillna("blue"),
    calima_weekly["calima_dai_flag"]  # leave as NaN (unknown) after coverage
)

section("CALIMA DAI FLAG — SUMMARY")
print("Heliyon coverage ends at week_start:", heliyon_end_week.date())
display(calima_weekly["calima_dai_flag"].value_counts(dropna=False).to_frame("weeks"))
display(calima_weekly.head(10))
display(calima_weekly.tail(10))


CALIMA DAI FLAG — SUMMARY
Heliyon coverage ends at week_start: 2022-03-28


,weeks
calima_dai_flag,
blue,279
NaN,144
red,19
green,16
yellow,13


,week_start,calima_dai_flag
0,2015-12-28,blue
1,2016-01-04,blue
2,2016-01-11,blue
3,2016-01-18,blue
4,2016-01-25,blue
5,2016-02-01,blue
6,2016-02-08,blue
7,2016-02-15,blue
8,2016-02-22,blue
9,2016-02-29,green


,week_start,calima_dai_flag
461,2024-10-28,NaN
462,2024-11-04,NaN
463,2024-11-11,NaN
464,2024-11-18,NaN
465,2024-11-25,NaN
466,2024-12-02,NaN
467,2024-12-09,NaN
468,2024-12-16,NaN
469,2024-12-23,NaN
470,2024-12-30,NaN


In [25]:
from pathlib import Path

out_dir = Path(r"C:\dev\projects\heat_mortality_analysis\data\interim\heliyon")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "heliyon_calima_dai_flag_weekly_tfe_2015w52_2024.parquet"
calima_weekly.to_parquet(out_path, index=False)

print("Wrote:", out_path)
print("Rows:", len(calima_weekly), "Cols:", calima_weekly.shape[1])

Wrote: C:\dev\projects\heat_mortality_analysis\data\interim\heliyon\heliyon_calima_dai_flag_weekly_tfe_2015w52_2024.parquet
Rows: 471 Cols: 2
